# Fair Lending Project - Seattle University


Building Model - Logistic Regression

## Libraries

In [14]:
import pandas as pd
import numpy as np
import sqlite3
import os
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, classification_report,
                             roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## Environment Setup

In [3]:
df_model_url='https://www.dropbox.com/scl/fi/fz71garcnma2vkrnpjh7u/modeling_dataset.csv?rlkey=ntgwlp7nxonr73ttek3t1qebb&st=dhuo7yp3&dl=1'
df_model_path = '../data/modeling_dataset.csv'

os.makedirs('../data', exist_ok=True)

#Cache data locally if not already present
if not os.path.exists(df_model_path):
    print("Downloading modeling_dataset.csv ...")
    response = requests.get(df_model_url)
    response.raise_for_status()
    with open(df_model_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded modeling_dataset.csv.")
else:
    print("modeling_dataset.csv already exists.")

Downloaded modeling_dataset.csv.


## Build Model

In [4]:
df_model = pd.read_csv(df_model_path)


print(f"df_model: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")
print(f"Default rate: {df_model['is_default'].mean():.3f}")

df_model: 886,649 rows × 36 columns
Default rate: 0.172


In [5]:
# Convert numerical columns to numeric type
should_be_numeric = ['approval_year', 'term', 'new_exist']
for col in should_be_numeric:
    if not pd.api.types.is_numeric_dtype(df_model[col]):
        print(f"Converting {col} to numeric...")
        df_model[col] = pd.to_numeric(df_model[col], errors='coerce')
        print(f"{col} dtype after: {df_model[col].dtype}")

In [6]:
# Define features
numeric_features = [
    'no_emp', 'gross_approved', 'sba_approved',
    'disbursement_gross','approval_year', 'term','new_exist',
    # Macro features (rename to match your DataFrame)
    'macro_unemployment', 'macro_fed_funds', 'macro_prime_rate',
    'macro_inflation', 'macro_bank_spread', 'macro_real_rate',
    # Demographic/segment/state features
    'segment_pct_women', 'segment_pct_minority', 'segment_pct_vet',
    'segment_avg_revenue', 'segment_avg_employees',
    'state_pct_women', 'state_pct_minority', 'state_pct_vet',
    'state_avg_revenue', 'state_avg_employees',
    # Engineered features
    'relative_emp_size', 'relative_emp_size_state',
    'loan_to_segment_rev_ratio', 'loan_to_state_rev_ratio'
]

categorical_features = [
    'state', 'sector_text', 'rev_line_cr', 'low_doc',
     'urban_rural', 'state_minority_density_group', 'minority_density_group'
]

df_model = df_model.drop(columns=['loan_id'])
df_model = df_model.dropna()

for col in categorical_features:
    df_model[col] = df_model[col].astype('category')

X = df_model[numeric_features + categorical_features].copy()
y = df_model['is_default'].copy()

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print(f"Train default rate: {y_train.mean():.3f}")
print(f"Test default rate:  {y_test.mean():.3f}")


Training set: 480,657 rows
Test set:     120,165 rows
Train default rate: 0.175
Test default rate:  0.175


In [11]:
df_model.describe()



,approval_year,term,no_emp,new_exist,gross_approved,sba_approved,disbursement_gross,is_default,segment_pct_women,segment_pct_minority,...,macro_unemployment,macro_fed_funds,macro_prime_rate,macro_inflation,macro_bank_spread,macro_real_rate,relative_emp_size,loan_to_segment_rev_ratio,relative_emp_size_state,loan_to_state_rev_ratio
count,600822.000000,600822.000000,600822.000000,600822.000000,6.008220e+05,6.008220e+05,6.008220e+05,600822.000000,600822.000000,600822.000000,...,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000,600822.000000
mean,2001.012736,103.602411,11.635052,1.270653,1.706182e+05,1.328384e+05,1.838861e+05,0.174627,1.645590,3.985276,...,5.707237,3.914486,6.850246,2.853908,2.934878,3.996951,3.283980,325.952917,2.676708,162.747471
std,6.284227,74.506459,73.480206,0.444297,2.591816e+05,2.115604e+05,2.686904e+05,0.379648,0.475497,0.136112,...,1.311882,2.109110,1.961516,0.927387,0.275859,1.696640,30.159579,942.039794,16.669327,253.069869
min,1988.000000,0.000000,0.000000,1.000000,1.000000e+03,5.000000e+02,4.000000e+03,0.000000,0.195621,1.228272,...,3.966667,0.089167,3.250000,-0.320000,1.660000,0.110000,0.000000,0.065947,0.000000,0.516923
25%,1995.000000,60.000000,2.000000,1.000000,3.000000e+04,1.750000e+04,3.942000e+04,0.000000,1.424628,3.912992,...,4.616667,1.927500,5.087500,2.600000,2.970000,2.820000,0.426438,32.028635,0.461202,27.770190
50%,2003.000000,84.000000,4.000000,1.000000,7.500000e+04,5.000000e+04,9.000000e+04,0.000000,1.676804,3.982952,...,5.541667,4.201667,7.138333,2.870000,2.990000,4.540000,1.034546,87.483509,1.002168,69.323508
75%,2006.000000,120.000000,10.000000,2.000000,2.000000e+05,1.510000e+05,2.143030e+05,0.000000,2.012923,4.077999,...,5.991667,5.298333,8.270833,3.220000,3.030000,5.330000,2.759021,259.872181,2.469549,188.216178
max,2014.000000,527.000000,9999.000000,2.000000,5.000000e+06,4.500000e+06,1.144632e+07,1.000000,2.748019,4.443380,...,9.608333,9.216667,10.873333,5.420000,3.160000,6.810000,9974.536692,45893.924833,2536.311179,6990.331726


## Logistics Regression Training

In [15]:
categorical_transformer = Pipeline([
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

# Model Training (Logistic Regression Baseline)
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

model_pipeline.fit(X_train, y_train)

# 6. Evaluation
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

feature_names = numeric_features + categorical_features

coeffs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model_pipeline.named_steps['classifier'].coef_[0]
}).sort_values(by='Coefficient', ascending=False)

print(" INFERENTIAL ANALYSIS: Top Drivers of Default")
print(coeffs.head(10))

print("\nDemographic Proxy Significance:")
print(coeffs[coeffs['Feature'].str.contains('pct_|density')])



              precision    recall  f1-score   support

           0       0.96      0.74      0.84     99181
           1       0.41      0.85      0.55     20984

    accuracy                           0.76    120165
   macro avg       0.68      0.80      0.70    120165
weighted avg       0.86      0.76      0.79    120165

ROC-AUC Score: 0.8727
 INFERENTIAL ANALYSIS: Top Drivers of Default
                      Feature  Coefficient
8             macro_fed_funds    13.195315
9            macro_prime_rate     7.129475
11          macro_bank_spread     1.311985
4               approval_year     1.175414
0                      no_emp     0.280538
1              gross_approved     0.240771
25  loan_to_segment_rev_ratio     0.176364
2                sba_approved     0.164778
20              state_pct_vet     0.128706
18            state_pct_women     0.104869

Demographic Proxy Significance:
                         Feature  Coefficient
20                 state_pct_vet     0.128706
18     

The baseline model demonstrates strong predictive performance with an ROC-AUC of 0.87, effectively catching 85% of defaults through a balanced weighting strategy that prioritizes risk detection. The most significant finding is the absolute dominance of macroeconomic indicators—specifically interest rates and bank spreads—as the primary drivers of loan failure, which far outweigh individual business characteristics. Crucially, the inferential analysis reveals that once these economic pressures and firm scales are controlled for, the high default disparity previously observed in minority-dense segments effectively disappears or even reverses slightly, suggesting that the risk is structural and economic rather than inherently linked to the demographic composition of the business segment.